# DecFM 2D Mixture — Results

Tables only. Every number is read from `../DecFM_2D/dr_mmd_flow/results/*.csv` and aggregated
in the house order: average within a (data seed, repeat), then within a data seed, then mean
± sd across the three seeds.

**Design.** 2-D Gaussian-mixture outcome, binary arm. Default cell: `n_obs = 20000`,
`M = 2000`, no-split nuisance, observational initialisation, doubly robust target.
Two samplers are run on identical particles from the same nuisance solve: `rbf_euler`
(RBF kernel, forward Euler) and `riesz_adamw` (Riesz kernel, AdamW with warm-up).

**How to read a ratio.** Every flow row is divided by the *count-matched* oracle floor — a
second oracle draw subsampled to the particle count. Raw values are not comparable across
rows with different M, because the floor itself moves with M.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import analysis_tables as at

ALL = at.load("DecFM_2D", estimator=None)
DR = ALL[ALL.estimator == "dr"]
BASE = DR[(DR.init == "observed")]
DEFAULT = ALL[(ALL.dr_fit == "nosplit") & (ALL.init == "observed")
              & (ALL.n_obs == 20000) & (ALL.n_particles_requested == 2000)]
SAMPLERS = ["rbf_euler", "riesz_adamw"]
SEEDS = sorted(int(s) for s in ALL.data_seed.unique())
print(f"{len(ALL)} rows   stages {sorted(ALL.tag.unique())}")
print(f"seeds {SEEDS}   samplers {SAMPLERS}   arms {sorted(ALL.target_a.unique())}   "
      f"estimators {sorted(ALL.estimator.unique())}")
print(f"default cell: {len(DEFAULT)} rows = {len(SEEDS)} seeds x 2 arms x 2 samplers x 3 estimators")

468 rows   stages ['est_crossfit', 'est_headline', 'est_init', 'est_nobs', 'est_particles']
seeds [0, 1, 2]   samplers ['rbf_euler', 'riesz_adamw']   arms [np.int64(0), np.int64(1)]   estimators ['dr', 'ipw', 'plugin']
default cell: 36 rows = 3 seeds x 2 arms x 2 samplers x 3 estimators


## 1. Headline

In [2]:
for s in SAMPLERS:
    g = DEFAULT[(DEFAULT.sampler == s) & (DEFAULT.estimator == "dr")]
    rows = [(label, [at.summarise(g, f"{m}_{k}") for m in ("sw2", "sw1", "mmd2")])
            for label, k in [("No flow (logged arm)", "initial"),
                             ("IMMD-Flow (ours)", "flow"),
                             ("Oracle floor (matched)", "floor_matched")]]
    print(at.plain_table(rows, ["SW2", "SW1", "MMD2"], first_column="Row", label_width=24,
                         title=f"DecFM_2D   sampler = {s}   (n_obs = 20000, M = 2000, "
                               f"seeds {SEEDS}, arms pooled)"))
    print()

DecFM_2D   sampler = rbf_euler   (n_obs = 20000, M = 2000, seeds [0, 1, 2], arms pooled)

Row                     SW2 ↓                SW1 ↓                MMD2 ↓
------------------------------------------------------------------------
No flow (logged arm)    0.51249 ± 0.00367    0.44012 ± 0.00569    0.04782 ± 0.00178
IMMD-Flow (ours)        0.11199 ± 0.01161    0.07953 ± 0.01236    0.00046 ± 0.00026
Oracle floor (matched)  0.07048 ± 0.01905    0.05433 ± 0.01644    0.00000 ± 0.00029

DecFM_2D   sampler = riesz_adamw   (n_obs = 20000, M = 2000, seeds [0, 1, 2], arms pooled)

Row                     SW2 ↓                SW1 ↓                MMD2 ↓
------------------------------------------------------------------------
No flow (logged arm)    0.51249 ± 0.00367    0.44012 ± 0.00569    0.04782 ± 0.00178
IMMD-Flow (ours)        0.12629 ± 0.01652    0.08540 ± 0.01659    0.00075 ± 0.00043
Oracle floor (matched)  0.07048 ± 0.01905    0.05433 ± 0.01644    0.00000 ± 0.00029



The flow closes most of the gap under both samplers: SW2 0.512 → 0.112
(`rbf_euler`) and → 0.126 (`riesz_adamw`), against a matched floor of 0.070, i.e. **1.59×**
and **1.79×** the floor.

## 2. Target-embedding estimator

In [3]:
for s in SAMPLERS:
    g = DEFAULT[DEFAULT.sampler == s]
    rows = [(label, [at.summarise(g[g.estimator == e], f"{m}_flow")
                     for m in ("sw2", "sw1", "mmd2")])
            for e, label in [("plugin", "Plug-in (CME only)"), ("ipw", "IPW (ratios only)"),
                             ("dr", "Doubly robust")]]
    print(at.plain_table(rows, ["SW2", "SW1", "MMD2"], first_column="Target embedding",
                         label_width=24, title=f"Estimator   sampler = {s}"))
    print()

Estimator   sampler = rbf_euler

Target embedding        SW2 ↓                SW1 ↓                MMD2 ↓
------------------------------------------------------------------------
Plug-in (CME only)      0.11305 ± 0.01507    0.07887 ± 0.01597    0.00033 ± 0.00036
IPW (ratios only)       0.15400 ± 0.02538    0.10551 ± 0.01486    0.00089 ± 0.00034
Doubly robust           0.11199 ± 0.01161    0.07953 ± 0.01236    0.00046 ± 0.00026

Estimator   sampler = riesz_adamw

Target embedding        SW2 ↓                SW1 ↓                MMD2 ↓
------------------------------------------------------------------------
Plug-in (CME only)      0.13997 ± 0.01763    0.09036 ± 0.01859    0.00071 ± 0.00047
IPW (ratios only)       0.18143 ± 0.02237    0.11902 ± 0.01625    0.00161 ± 0.00059
Doubly robust           0.12629 ± 0.01652    0.08540 ± 0.01659    0.00075 ± 0.00043



Plug-in and DR are indistinguishable; **IPW is clearly worse on both samplers**
(+37% SW2 under Euler, +44% under AdamW). This is the design that separates the estimator
axis most cleanly — ColorMNIST does not separate it at any sample size and CelebA is
algebraically degenerate there.

## 3. Per arm

In [4]:
print(f"  {'sampler':13s} {'arm':>4s} {'logged':>9s} {'SW2':>9s} {'floor':>9s} {'ratio':>7s}")
for s in SAMPLERS:
    for a in (0, 1):
        g = DEFAULT[(DEFAULT.sampler == s) & (DEFAULT.estimator == "dr") & (DEFAULT.target_a == a)]
        lg, _ = at.summarise(g, "sw2_initial")
        fl, _ = at.summarise(g, "sw2_flow")
        fo, _ = at.summarise(g, "sw2_floor_matched")
        print(f"  {s:13s} {a:4d} {lg:9.5f} {fl:9.5f} {fo:9.5f} {fl / fo:6.2f}x")

  sampler        arm    logged       SW2     floor   ratio
  rbf_euler        0   0.51612   0.12291   0.07048   1.74x
  rbf_euler        1   0.50885   0.10108   0.07048   1.43x
  riesz_adamw      0   0.51612   0.14399   0.07048   2.04x
  riesz_adamw      1   0.50885   0.10860   0.07048   1.54x


Arm 1 is the easier arm under both samplers, by roughly 0.3 of ratio.

## 4. Ablations

One helper, four sweeps. Each row shows both samplers side by side against the floor at that
row's own particle count.

In [5]:
def sweep(frame, by, order, labels, title):
    print(title)
    print(f"  {'':18s} {'rbf_euler':>19s}   {'riesz_adamw':>19s}   {'floor':>9s}")
    print(f"  {'':18s} {'SW2':>9s} {'ratio':>7s}   {'SW2':>9s} {'ratio':>7s}")
    for value in order:
        g = frame[frame[by] == value]
        out = []
        for s in SAMPLERS:
            gs = g[g.sampler == s]
            m, _ = at.summarise(gs, "sw2_flow")
            fo, _ = at.summarise(gs, "sw2_floor_matched")
            out.append((m, m / fo))
        fo, _ = at.summarise(g, "sw2_floor_matched")
        print(f"  {labels[value]:18s} {out[0][0]:9.5f} {out[0][1]:6.2f}x   "
              f"{out[1][0]:9.5f} {out[1][1]:6.2f}x   {fo:9.5f}")
    print()


NOSPLIT = BASE[BASE.dr_fit == "nosplit"]
sweep(NOSPLIT[NOSPLIT.n_particles_requested == 2000], "n_obs", [5000, 10000, 15000, 20000],
      {n: f"n_obs = {n}" for n in (5000, 10000, 15000, 20000)},
      "Observations   (M = 2000, no-split)")
sweep(NOSPLIT[NOSPLIT.n_obs == 20000], "n_particles_requested", [250, 500, 1000, 2000],
      {m: f"M = {m}" for m in (250, 500, 1000, 2000)},
      "Particles   (n_obs = 20000, no-split)")
sweep(BASE[(BASE.n_obs == 20000) & (BASE.n_particles_requested == 2000)], "dr_fit",
      ["nosplit", "crossfit"], {"nosplit": "No-split", "crossfit": "Cross-fitted"},
      "Nuisance fitting   (n_obs = 20000, M = 2000)")
sweep(DR[(DR.dr_fit == "nosplit") & (DR.n_obs == 20000) & (DR.n_particles_requested == 2000)],
      "init", ["observed", "moment", "normal"],
      {"observed": "Observed (P_Y)", "moment": "Moment-matched", "normal": "Standard normal"},
      "Initialisation   (n_obs = 20000, M = 2000)")

Observations   (M = 2000, no-split)
                               rbf_euler           riesz_adamw       floor
                           SW2   ratio         SW2   ratio
  n_obs = 5000         0.13716   1.95x     0.14652   2.08x     0.07048


  n_obs = 10000        0.11062   1.57x     0.12560   1.78x     0.07048
  n_obs = 15000        0.12134   1.72x     0.12785   1.81x     0.07048


  n_obs = 20000        0.11199   1.59x     0.12629   1.79x     0.07048

Particles   (n_obs = 20000, no-split)
                               rbf_euler           riesz_adamw       floor
                           SW2   ratio         SW2   ratio
  M = 250              0.18885   1.01x     0.19499   1.04x     0.18693
  M = 500              0.16240   1.29x     0.17019   1.35x     0.12626
  M = 1000             0.14406   1.42x     0.15512   1.53x     0.10137
  M = 2000             0.11199   1.59x     0.12629   1.79x     0.07048

Nuisance fitting   (n_obs = 20000, M = 2000)
                               rbf_euler           riesz_adamw       floor
                           SW2   ratio         SW2   ratio


  No-split             0.11199   1.59x     0.12629   1.79x     0.07048
  Cross-fitted         0.11301   1.60x     0.12576   1.78x     0.07048

Initialisation   (n_obs = 20000, M = 2000)
                               rbf_euler           riesz_adamw       floor
                           SW2   ratio         SW2   ratio


  Observed (P_Y)       0.11199   1.59x     0.12629   1.79x     0.07048
  Moment-matched       0.10975   1.56x     0.15014   2.13x     0.07048
  Standard normal      0.13408   1.90x     0.45490   6.45x     0.07048



**Observations.** Improves from 1.95× to 1.59× (Euler) and 2.08× to 1.79×
(AdamW) over a 4× increase in data, with the gain mostly spent by n_obs = 10000.

**Particles.** The raw column falls (0.189 → 0.112) while the ratio *rises* (1.01× → 1.59×),
because the matched floor falls faster than the flow does. Read the ratio, not the raw value:
small clouds look good only because an independent draw of the same size also looks bad.

**Nuisance fitting.** No difference (1.59 vs 1.60 Euler, 1.79 vs 1.78 AdamW). Cross-fitting
neither helps nor hurts at this overlap.

**Initialisation — the one place the samplers diverge.** `rbf_euler` is robust: a standard
normal start costs 1.59× → 1.90×. `riesz_adamw` is not: the same change costs 1.79× →
**6.45×**, a 3.6× degradation in SW2 (0.126 → 0.455). AdamW sets its step from the gradient's
running RMS rather than its magnitude, so a distant start does not converge within the 2000
iterations the other settings use. Euler, whose step follows the gradient, does.

## 5. Coverage

In [6]:
print(at.plain_table(
    [(tag, [(len(g), 0.0), (len(g) / 12, 0.0)]) for tag, g in ALL.groupby("tag", observed=True)],
    ["rows", "cells"], first_column="Launcher stage", label_width=18, digits=0, arrow=False,
    title="DecFM_2D   what is on disk",
    note="cells = rows / (2 arms x 2 samplers x 3 estimators); 3 seeds each."))
print(f"\ntotal {len(ALL)} rows, all five stages complete at {len(SEEDS)} seeds.")

DecFM_2D   what is on disk

Launcher stage    rows                 cells
--------------------------------------------
est_crossfit      144 ± 0              12 ± 0
est_headline      36 ± 0               3 ± 0
est_init          72 ± 0               6 ± 0
est_nobs          108 ± 0              9 ± 0
est_particles     108 ± 0              9 ± 0

cells = rows / (2 arms x 2 samplers x 3 estimators); 3 seeds each.

total 468 rows, all five stages complete at 3 seeds.
